# Génération live Nacholmo v2 — pipeline corrigé

Ce notebook génère réellement sur la RTX. Le brut est créé en **text2img**, comme dans la fiche Nacholmo, puis une pipeline **img2img DDIM distincte** exécute les 100 pas SRPG. E008 reste un témoin de scannabilité, mais son Stage-1 img2img à 1,60 n'est plus utilisé pour le rendu artistique.

## 0. Demande utilisateur et paramètres reproductibles

In [ ]:
PAYLOAD = "https://example.prooftag.test/t/e007-fixed"  # témoin E008 exact
PROMPT = (
    "one coherent lush botanical garden illustration, broad flowing leaves and flowers, "
    "organic curves, elegant composition, detailed editorial art"
)
SEED = 2026  # témoin strict 26/26 observé dans E008
ERROR_CORRECTION = "H"
CONTROLNET_MODEL_ID = "Nacholmo/controlnet-qr-pattern-v2"
BASE_MODEL_ID = "Nacholmo/Counterfeit-V2.5-vae-swapped"  # recommande par l'auteur
OFFICIAL_BASE_MODEL_ID = "runwayml/stable-diffusion-v1-5"  # fiche officielle
RAW_STEPS = 30
GUIDANCE_SCALE = 6.5
RAW_PROFILES = (
    {"name": "art", "scale": 0.40, "control_end": 0.55},
    {"name": "balanced", "scale": 0.55, "control_end": 0.70},
    {"name": "structured", "scale": 0.75, "control_end": 0.85},
)
RAW_SELECTED_PROFILE = "balanced"  # aucun choix automatique cache
SRPG_STEPS = 100
SRPG_STRENGTH = 1.00
SRPG_CONTROLNET_SCALE = 1.60
SRPG_QR_WEIGHT = 500.0
SRPG_PERCEPTUAL_WEIGHT = 3.0
SRPG_FUNCTIONAL_WEIGHT = 4.0
SRPG_CENTER_FRACTION = 1 / 3
SRPG_DARK_THRESHOLD = 0.50
SRPG_LIGHT_THRESHOLD = 0.50
SRPG_MAX_NOISE_DELTA_RMS = 2.0
SRPG_ETA = 0.0
DISPLAY_EVERY = 1  # 1 = chaque pas; 5 = affichage et stockage plus légers

## 1. Kernel GPU, imports et dossier persistant

In [ ]:
import csv
import gc
import json
import math
import shutil
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Markdown, display
from PIL import Image, ImageOps

from prooftag_qr.backends import ControlNetBackend
from prooftag_qr.config import Settings
from prooftag_qr.experiments import NEGATIVE_PROMPT_PROFILES
from prooftag_qr.qr import generate_qr, module_error_rate
from prooftag_qr.quality import image_change_metrics
from prooftag_qr.quality_scoring import CLIPQualityScorer
from prooftag_qr.schemas import GenerationRequest
from prooftag_qr.srpg import SRPGConfig, run_srpg_controlnet_img2img
from prooftag_qr.validation import QRValidator, summarize_validation_records

if not torch.cuda.is_available():
    raise RuntimeError("Utiliser le kernel du serveur RTX, pas Python Windows.")
run_name = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + f"-nacholmo-seed-{SEED}"
run_dir = Path("/data/notebook-runs") / run_name
run_dir.mkdir(parents=True, exist_ok=False)
display(Markdown(f"**GPU :** `{torch.cuda.get_device_name(0)}`  \
**Résultats :** `{run_dir}`"))

## 2. Stage-1 correct : ControlNet text2img, sans QR comme image initiale

In [ ]:
negative_prompt = NEGATIVE_PROMPT_PROFILES["structure_safe"]
raw_settings = Settings(
    data_dir=Path("/data"),
    model_cache_dir=Path("/cache"),
    default_backend="controlnet",
    base_model_id=BASE_MODEL_ID,
    controlnet_model_id=CONTROLNET_MODEL_ID,
    controlnet_model_subfolder="",
    controlnet_conditioning_profile="nacholmo_extremes_25",
    controlnet_pipeline_mode="text2img",
    device="cuda",
    srpg_enabled=False,
    guided_rediffusion_enabled=False,
    latent_refinement_enabled=False,
)
request = GenerationRequest(
    payload=PAYLOAD,
    prompt=PROMPT,
    negative_prompt=negative_prompt,
    backend="controlnet",
    error_correction=ERROR_CORRECTION,
    seed=SEED,
    steps=RAW_STEPS,
    strength=1.0,  # ignore en text2img
    guidance_scale=GUIDANCE_SCALE,
    controlnet_scale=next(p["scale"] for p in RAW_PROFILES if p["name"] == RAW_SELECTED_PROFILE),
    max_attempts=1,
)
raw_backend = ControlNetBackend(raw_settings)
raw_pipeline = raw_backend._load()
from diffusers import DPMSolverMultistepScheduler
raw_pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
    raw_pipeline.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True
)
display(Markdown(
    f"**Base :** `{raw_settings.base_model_id}` — **ControlNet :** `{raw_settings.controlnet_model_id}`  \
"
    f"**Pipeline :** `{raw_pipeline.__class__.__name__}` / `{raw_pipeline.scheduler.__class__.__name__}`  \
"
    f"**Stage-1 text2img :** {RAW_STEPS} pas, CFG {GUIDANCE_SCALE:.1f}, profils {RAW_PROFILES}  \
"
    f"**SRPG :** {SRPG_STEPS} pas, ControlNet {SRPG_CONTROLNET_SCALE:.2f}"
))

## 3. Condition 25/25 et trois compromis force / fin de guidage

In [ ]:
blueprint = generate_qr(PAYLOAD, ERROR_CORRECTION, size=512)
blueprint.image.save(run_dir / "00_qr_control.png")
control_image = raw_backend.control_image(blueprint)
control_image.save(run_dir / "00_controlnet_condition.png")
torch.cuda.reset_peak_memory_stats()
raw_candidates = {}
for profile in RAW_PROFILES:
    generator = torch.Generator(device=raw_settings.device).manual_seed(SEED)
    candidate = raw_pipeline(
        prompt=PROMPT,
        negative_prompt=negative_prompt,
        image=control_image,
        width=512,
        height=512,
        num_inference_steps=RAW_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=profile["scale"],
        control_guidance_start=0.0,
        control_guidance_end=profile["control_end"],
        generator=generator,
    ).images[0].convert("RGB")
    raw_candidates[profile["name"]] = candidate
    candidate.save(run_dir / f"01_nacholmo_raw_{profile['name']}.png")
raw = raw_candidates[RAW_SELECTED_PROFILE]
raw.save(run_dir / "01_nacholmo_raw.png")
raw_error = module_error_rate(raw, blueprint)
peak_cuda_mib = torch.cuda.max_memory_allocated() / 1024**2
fig, axes = plt.subplots(1, 2 + len(raw_candidates), figsize=(5 * (2 + len(raw_candidates)), 5))
for axis, title, image in zip(
    axes,
    ["QR de contrôle", "Condition 25/25"] + [
        f"{profile['name']} s={profile['scale']:.2f} end={profile['control_end']:.2f}"
        for profile in RAW_PROFILES
    ],
    [blueprint.image, control_image] + list(raw_candidates.values()),
    strict=True,
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
display(Markdown(
    f"QR v{blueprint.version}, {blueprint.matrix.shape[0]}×{blueprint.matrix.shape[1]} modules — "
    f"candidat retenu **{RAW_SELECTED_PROFILE}**, erreur module **{raw_error:.3%}**, "
    f"pic CUDA **{peak_cuda_mib:.0f} MiB**."
))

## 4. Validation externe du brut par décodeur et scénario

In [ ]:
validator = QRValidator()


def validate_candidate(image, filename):
    records = validator.validate(image, PAYLOAD)
    summary = summarize_validation_records(records)
    passed = sum(record.exact_payload_match for record in records)
    (run_dir / filename).write_text(
        json.dumps([asdict(record) for record in records], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return records, summary, passed


def show_validation(name, records, summary, passed):
    decoder_text = " — ".join(
        f"{decoder}: {rate:.1%}"
        for decoder, rate in summary["decoder_pass_rates"].items()
    )
    failed = [f"{record.decoder}/{record.scenario}" for record in records if not record.exact_payload_match]
    display(Markdown(
        f"**{name} : {passed}/{len(records)}** — {decoder_text}  \
"
        f"Échecs : `{', '.join(failed) if failed else 'aucun'}`"
    ))


raw_sweep_validation = {}
for profile_name, candidate in raw_candidates.items():
    records, summary, passed = validate_candidate(
        candidate, f"01_nacholmo_raw_{profile_name}.validations.json"
    )
    raw_sweep_validation[profile_name] = {"passed": passed, "summary": summary}
    show_validation(f"Nacholmo text2img {profile_name}", records, summary, passed)
raw_records, raw_validation, raw_passed = validate_candidate(
    raw, "01_nacholmo_raw.validations.json"
)

## 5. Pipeline img2img séparée et diffusion guidée SRPG, pas par pas

À gauche : prédiction propre `x0`. À droite : modules actifs dans la loss. Chaque aperçu est réellement recalculé et enregistré.

In [ ]:
# Libère entièrement la pipeline text2img avant de charger la pipeline SRPG img2img.
del raw_pipeline
raw_backend._pipeline = None
del raw_backend
gc.collect()
torch.cuda.empty_cache()
srpg_settings = Settings(
    data_dir=Path("/data"),
    model_cache_dir=Path("/cache"),
    default_backend="controlnet",
    base_model_id=BASE_MODEL_ID,
    controlnet_model_id=CONTROLNET_MODEL_ID,
    controlnet_conditioning_profile="binary",
    controlnet_pipeline_mode="img2img",
    device="cuda",
    srpg_enabled=False,
    guided_rediffusion_enabled=False,
    latent_refinement_enabled=False,
)
srpg_backend = ControlNetBackend(srpg_settings)
pipeline = srpg_backend._load()
display(Markdown(
    f"**Pipeline SRPG :** `{pipeline.__class__.__name__}` / `{pipeline.scheduler.__class__.__name__}`"
))
history = []
status_handle = display(HTML("<b>Initialisation SRPG…</b>"), display_id=True)
image_handle = display(Image.new("RGB", (1024, 512), "white"), display_id=True)


def show_srpg_step(preview, step):
    history.append(step)
    error_rgb = ImageOps.colorize(
        preview.active_module_map, black="black", white="red"
    ).convert("RGB")
    panel = Image.new("RGB", (1024, 512), "white")
    panel.paste(preview.predicted_clean_image.resize((512, 512)), (0, 0))
    panel.paste(error_rgb.resize((512, 512)), (512, 0))
    preview.predicted_clean_image.save(
        run_dir / f"02_srpg_step_{step.index:03d}_x0.png"
    )
    preview.active_module_map.save(
        run_dir / f"02_srpg_step_{step.index:03d}_errors.png"
    )
    status_handle.update(HTML(
        f"<b>Pas {step.index + 1}/{SRPG_STEPS}</b> — timestep {step.timestep} — "
        f"module {step.module_error_rate:.3%} — SRL {step.scanning_robust_loss:.4f} — "
        f"LPIPS {step.perceptual_loss:.4f} — Δ bruit {step.noise_delta_rms:.4f}"
    ))
    image_handle.update(panel)


srpg_generator = torch.Generator(device=srpg_settings.device).manual_seed(
    (SEED + srpg_settings.srpg_seed_offset) % (2**32)
)
srpg = run_srpg_controlnet_img2img(
    pipeline,
    raw,
    blueprint,
    prompt=PROMPT,
    negative_prompt=negative_prompt,
    guidance_scale=GUIDANCE_SCALE,
    generator=srpg_generator,
    control_image=srpg_backend.control_image(blueprint),
    config=SRPGConfig(
        steps=SRPG_STEPS,
        strength=SRPG_STRENGTH,
        controlnet_scale=SRPG_CONTROLNET_SCALE,
        qr_weight=SRPG_QR_WEIGHT,
        perceptual_weight=SRPG_PERCEPTUAL_WEIGHT,
        functional_weight=SRPG_FUNCTIONAL_WEIGHT,
        center_fraction=SRPG_CENTER_FRACTION,
        dark_threshold=SRPG_DARK_THRESHOLD,
        light_threshold=SRPG_LIGHT_THRESHOLD,
        target_module_error_rate=0.0,
        max_noise_delta_rms=SRPG_MAX_NOISE_DELTA_RMS,
        eta=SRPG_ETA,
        max_mean_absolute_change=0.40,
        min_relative_module_improvement=0.0,
        save_step_previews=True,
        preview_interval=DISPLAY_EVERY,
    ),
    preview_callback=show_srpg_step,
)
srpg.image.save(run_dir / "03_nacholmo_srpg.png")
with (run_dir / "03_nacholmo_srpg.steps.csv").open(
    "w", newline="", encoding="utf-8"
) as stream:
    writer = csv.DictWriter(stream, fieldnames=list(asdict(srpg.steps[0]).keys()))
    writer.writeheader()
    writer.writerows(asdict(step) for step in srpg.steps)
display(Markdown(
    f"**SRPG terminé :** module {srpg.initial_module_error_rate:.3%} → "
    f"{srpg.final_module_error_rate:.3%}; MAE {srpg.mean_absolute_change:.4f}; "
    f"porte interne `{srpg.accepted}` (`{srpg.rejection_reason or 'accepté'}`)."
))

## 6. Courbes et validation externe de la sortie SRPG

In [ ]:
steps = list(srpg.steps)
x = [step.index for step in steps]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
series = [
    ("module_error_rate", "Erreur module"),
    ("scanning_robust_loss", "Scanning Robust Loss"),
    ("perceptual_loss", "LPIPS"),
    ("noise_delta_rms", "Delta bruit RMS"),
]
for axis, (field, title) in zip(axes.flat, series, strict=True):
    axis.plot(x, [getattr(step, field) for step in steps])
    axis.set_title(title)
    axis.set_xlabel("Pas DDIM")
    axis.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(run_dir / "04_srpg_curves.png", dpi=140)
srpg_records, srpg_validation, srpg_passed = validate_candidate(
    srpg.image, "03_nacholmo_srpg.validations.json"
)
show_validation("Nacholmo + SRPG", srpg_records, srpg_validation, srpg_passed)
display(srpg.image)

## 7. Candidats directs et réparés, avec CLIP-aesthetic/CLIPScore

La réparation déterministe reste visible pour diagnostic. Elle ne devient livraison que si elle réussit 26/26, puis elle est comparée aux sorties directes par qualité.

In [ ]:
quality_scorer = CLIPQualityScorer(srpg_settings.model_cache_dir)
candidates = []


def evaluate(name, image):
    records, validation, passed = validate_candidate(
        image, f"05_{name}.validations.json"
    )
    change = image_change_metrics(image, raw)
    try:
        quality = quality_scorer.score(image, PROMPT)
        clip_aesthetic = quality.clip_aesthetic
        clip_score = quality.clip_score
    except Exception as exc:
        print(f"Scores CLIP indisponibles pour {name}: {exc!r}")
        clip_aesthetic = None
        clip_score = None
    row = {
        "name": name,
        "image": image,
        "passed": passed,
        "validations": len(records),
        "strict_all": passed == len(records),
        "pass_rate": passed / len(records),
        "original_pass_rate": validation["scenario_pass_rates"].get("original", 0.0),
        "worst_decoder_pass_rate": validation["worst_decoder_pass_rate"],
        "decoder_pass_rates": validation["decoder_pass_rates"],
        "module_error_rate": module_error_rate(image, blueprint),
        "clip_aesthetic": clip_aesthetic,
        "clip_score": clip_score,
        **change,
    }
    candidates.append(row)
    image.save(run_dir / f"05_{name}.png")
    print(
        f"{name:36s} scan={passed:2d}/{len(records)} "
        f"weak={validation['worst_decoder_pass_rate']:.1%} "
        f"aes={clip_aesthetic if clip_aesthetic is not None else float('nan'):.3f} "
        f"clip={clip_score if clip_score is not None else float('nan'):.3f}"
    )


for profile_name, candidate in raw_candidates.items():
    evaluate(f"raw_{profile_name}", candidate)
evaluate("srpg", srpg.image)
repair_backend = srpg_backend
for source_name, source_image in (("srpg", srpg.image), ("raw", raw)):
    for variant_name, variant_image in repair_backend.variants(
        source_image, blueprint, request=request, seed=SEED
    ):
        if variant_name == "raw":
            continue
        evaluate(f"{source_name}_{variant_name}", variant_image)

## 8. Porte de livraison : 26/26 obligatoire, puis esthétique

In [ ]:
def quality_value(value):
    return float(value) if value is not None and not math.isnan(float(value)) else float("-inf")


strict_candidates = [row for row in candidates if row["strict_all"]]
if strict_candidates:
    selected = max(
        strict_candidates,
        key=lambda row: (
            quality_value(row["clip_aesthetic"]),
            quality_value(row["clip_score"]),
            -row["mean_absolute_change"],
        ),
    )
    delivery_status = "DELIVERY"
    output_path = run_dir / "06_DELIVERY.png"
    message = "CANDIDAT AUTOMATIQUE 26/26 — validation physique encore obligatoire"
else:
    selected = max(
        candidates,
        key=lambda row: (
            row["worst_decoder_pass_rate"],
            row["original_pass_rate"],
            row["pass_rate"],
            quality_value(row["clip_aesthetic"]),
        ),
    )
    delivery_status = "REJECTED"
    output_path = run_dir / "06_BEST_OBSERVED_NOT_DELIVERABLE.png"
    message = "REJET : aucune image 26/26"
selected["image"].save(output_path)
display(Markdown(
    f"## {message}  \
Sélection : **{selected['name']}**, "
    f"{selected['passed']}/{selected['validations']}, "
    f"CLIP-aesthetic {selected['clip_aesthetic']}, CLIPScore {selected['clip_score']}."
))
display(selected["image"])

## 9. Comparaison, manifeste, validation téléphone et archive

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for axis, title, image in zip(
    axes,
    ["Contrôle QR", "Nacholmo brut", "Nacholmo + SRPG", selected["name"]],
    [blueprint.image, raw, srpg.image, selected["image"]],
    strict=True,
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
fig.savefig(run_dir / "07_comparison.png", dpi=140)

serializable_candidates = [
    {key: value for key, value in row.items() if key != "image"}
    for row in candidates
]
manifest = {
    "run_name": run_name,
    "timestamp": datetime.now(UTC).isoformat(),
    "model_id": CONTROLNET_MODEL_ID,
    "base_model_id": BASE_MODEL_ID,
    "method_sources": [
        "https://huggingface.co/Nacholmo/controlnet-qr-pattern-v2",
        "https://huggingface.co/docs/diffusers/api/pipelines/controlnet",
        "https://huggingface.co/Nacholmo/Counterfeit-V2.5-vae-swapped",
        "https://huggingface.co/Nacholmo/controlnet-qr-pattern-v2/discussions/1",
    ],
    "payload": PAYLOAD,
    "prompt": PROMPT,
    "seed": SEED,
    "parameters": {
        "raw_pipeline_mode": "text2img",
        "raw_scheduler": "DPMSolverMultistepScheduler/dpmsolver++/Karras",
        "raw_steps": RAW_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "raw_conditioning_profile": "nacholmo_extremes_25",
        "raw_profiles": RAW_PROFILES,
        "raw_selected_profile": RAW_SELECTED_PROFILE,
        "srpg_pipeline_mode": "img2img",
        "srpg_steps": SRPG_STEPS,
        "srpg_strength": SRPG_STRENGTH,
        "srpg_controlnet_scale": SRPG_CONTROLNET_SCALE,
        "srpg_qr_weight": SRPG_QR_WEIGHT,
        "srpg_perceptual_weight": SRPG_PERCEPTUAL_WEIGHT,
        "srpg_functional_weight": SRPG_FUNCTIONAL_WEIGHT,
        "srpg_center_fraction": SRPG_CENTER_FRACTION,
        "srpg_dark_threshold": SRPG_DARK_THRESHOLD,
        "srpg_light_threshold": SRPG_LIGHT_THRESHOLD,
        "srpg_max_noise_delta_rms": SRPG_MAX_NOISE_DELTA_RMS,
        "srpg_eta": SRPG_ETA,
    },
    "raw_validation": raw_validation,
    "raw_sweep_validation": raw_sweep_validation,
    "srpg_validation": srpg_validation,
    "srpg_internal_accepted": srpg.accepted,
    "srpg_rejection_reason": srpg.rejection_reason,
    "delivery_status": delivery_status,
    "selected_variant": selected["name"],
    "candidates": serializable_candidates,
}
(run_dir / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
with (run_dir / "candidates.csv").open("w", newline="", encoding="utf-8") as stream:
    fields = [
        "name", "passed", "validations", "strict_all", "pass_rate",
        "original_pass_rate", "worst_decoder_pass_rate", "module_error_rate",
        "clip_aesthetic", "clip_score", "changed_pixel_ratio",
        "mean_absolute_change",
    ]
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows({key: row[key] for key in fields} for row in candidates)
if strict_candidates:
    with (run_dir / "phone-validation.csv").open(
        "w", newline="", encoding="utf-8"
    ) as stream:
        fields = ["device", "protocol", "attempts", "successes", "exact_payload", "notes"]
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for protocol in (
            "screen-front-30cm", "screen-angle-30deg", "screen-low-light",
            "print-5cm", "print-angle-30deg",
        ):
            writer.writerow({
                "device": "", "protocol": protocol, "attempts": 10,
                "successes": "", "exact_payload": "", "notes": "",
            })
archive = shutil.make_archive(
    str(Path("/workspace/results") / run_name),
    "gztar",
    root_dir=run_dir.parent,
    base_dir=run_name,
)
display(Markdown(
    f"**Archive :** `results/{Path(archive).name}`  \
"
    f"**Dossier persistant :** `{run_dir}`"
))